In [2]:
from datasets import load_dataset

tool_call_dataset = load_dataset("glaiveai/glaive-function-calling-v2")

/Users/sauravprateek/Documents/saurav-codes/neural-nets-codelab/saurav-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
tool_call_dataset

DatasetDict({
    train: Dataset({
        features: ['system', 'chat'],
        num_rows: 112960
    })
})

In [29]:
print(tool_call_dataset['train'][101]['chat'])

USER: Hi, I need to convert 500 US dollars to Euros. Can you help me with that?


ASSISTANT: <functioncall> {"name": "convert_currency", "arguments": '{"amount": 500, "from_currency": "USD", "to_currency": "EUR"}'} <|endoftext|>


FUNCTION RESPONSE: {"converted_amount": 425.50, "from_currency": "USD", "to_currency": "EUR"}


ASSISTANT: Sure, 500 US dollars is approximately 425.50 Euros. <|endoftext|>





In [5]:
train_size = 0.85
train_test_split_dataset = tool_call_dataset['train'].train_test_split(train_size=train_size)

In [6]:
train_test_split_dataset

DatasetDict({
    train: Dataset({
        features: ['system', 'chat'],
        num_rows: 96016
    })
    test: Dataset({
        features: ['system', 'chat'],
        num_rows: 16944
    })
})

In [46]:
def process_dataset(dataset_split):
    functionless = 0
    agent_response_none = 0
    prompts = []

    for row in dataset_split:
        system_instructions = row['system']
        chat = row['chat']
        user_agent_responses = []

        if system_instructions.strip() == 'SYSTEM: You are a helpful assistant, with no access to external functions.':
            functionless += 1
            chats = chat.split('\n')
            user_agent_responses.append(chats[0])
            user_agent_responses.append(
                'ASSISTANT: I am a Function calling model, so you will have to provide me with some external function. <|endoftext|>'
            )
        else:
            chats = chat.split('\n')
            function_call_responses = []
            for chat_row in chats:
                if not chat_row.strip():
                    continue
                if chat_row.strip().startswith('FUNCTION RESPONSE:'):
                    break
                function_call_responses.append(chat_row)
            
            user_agent_responses.extend(function_call_responses)
            if not user_agent_responses:
                agent_response_none += 1

        finalized_prompt = system_instructions + '\n\n' + '\n'.join(user_agent_responses)
        prompts.append(finalized_prompt)

    percentage = (functionless * 100.0) / dataset_split.num_rows
    function_call_rows = dataset_split.num_rows - functionless
    no_assistant_func_response_percentage = (agent_response_none*100.0) / function_call_rows
    print(f'Number of rows with no provided functions: {functionless} / {dataset_split.num_rows} ({percentage:.2f}%)')
    print(
        f'Number of rows with no assistant function response: {agent_response_none} / {function_call_rows} ({no_assistant_func_response_percentage:.2f}%)')

    return prompts

In [47]:
prepared_train_prompts = process_dataset(train_test_split_dataset['train'])

Number of rows with no provided functions: 29431 / 96016 (30.65%)
Number of rows with no assistant function response: 0 / 66585 (0.00%)


In [48]:
len(prepared_train_prompts)

96016

In [52]:
print(prepared_train_prompts[20001])

SYSTEM: You are a helpful assistant with access to the following functions. Use them if required -
{
    "name": "find_nearest_gas_station",
    "description": "Find the nearest gas station based on current location",
    "parameters": {
        "type": "object",
        "properties": {
            "latitude": {
                "type": "number",
                "description": "The latitude of the current location"
            },
            "longitude": {
                "type": "number",
                "description": "The longitude of the current location"
            }
        },
        "required": [
            "latitude",
            "longitude"
        ]
    }
}


USER: I'm running low on gas. Can you help me find the nearest gas station?
ASSISTANT: Of course, I can help with that. Could you please provide me with your current latitude and longitude? <|endoftext|>
USER: Sure, my latitude is 40.7128 and my longitude is -74.0060.
ASSISTANT: <functioncall> {"name": "find_nearest_ga

In [54]:
from transformers import AutoTokenizer

gemma_270m = 'google/gemma-3-270m'
tokenizer = AutoTokenizer.from_pretrained(gemma_270m)

In [53]:
# Running method in parallel and hence processing individual elements. 
def format_and_tokenize_single_dataset(dataset_split):
    tokenized_dataset = tokenizer.encode(dataset_split)
    return tokenized_dataset

In [55]:
from joblib import Parallel, delayed
from tqdm.auto import tqdm

tasks = (delayed(format_and_tokenize_single_dataset)(row) for row in prepared_train_prompts)

# Run in parallel
tokenized_dataset = list(tqdm(
    Parallel(n_jobs=-1, return_as="generator", batch_size=1000)(tasks), 
    total=len(prepared_train_prompts), 
    desc="Tokenizing"
))

Tokenizing: 100%|██████████| 96016/96016 [01:46<00:00, 903.60it/s] 


In [58]:
print(tokenizer.decode(tokenized_dataset[5101]))

<bos>SYSTEM: You are a helpful assistant with access to the following functions. Use them if required -
{
    "name": "search_recipes",
    "description": "Search for recipes based on ingredients",
    "parameters": {
        "type": "object",
        "properties": {
            "ingredients": {
                "type": "array",
                "items": {
                    "type": "string"
                },
                "description": "The list of ingredients to search for"
            }
        },
        "required": [
            "ingredients"
        ]
    }
}


USER: Hey, I have some ingredients in my fridge and I don't know what to cook with them. Can you help me?
ASSISTANT: Of course, I'd be happy to help! Please tell me what ingredients you have. <|endoftext|>
USER: I have chicken, bell peppers, and rice.
ASSISTANT: <functioncall> {"name": "search_recipes", "arguments": '{"ingredients": ["chicken", "bell peppers", "rice"]}'} <|endoftext|>


In [67]:
print(tokenized_dataset[101])

[2, 90846, 236787, 1599, 659, 496, 11045, 16326, 607, 2802, 531, 506, 2269, 5151, 236761, 6890, 1091, 768, 3149, 753, 107, 236782, 107, 140, 236775, 1201, 1083, 623, 2305, 236779, 21000, 827, 107, 140, 236775, 7777, 1083, 623, 8980, 573, 496, 6387, 1699, 1061, 3822, 827, 107, 140, 236775, 19031, 1083, 642, 107, 144, 236775, 2084, 1083, 623, 5973, 827, 107, 144, 236775, 15921, 1083, 642, 107, 148, 236775, 21000, 236779, 3250, 1083, 642, 107, 152, 236775, 2084, 1083, 623, 2383, 827, 107, 152, 236775, 7777, 1083, 623, 818, 3822, 529, 506, 6387, 236775, 107, 148, 236783, 107, 144, 1263, 107, 144, 236775, 15979, 1083, 870, 107, 148, 236775, 21000, 236779, 3250, 236775, 107, 144, 236842, 107, 140, 236783, 107, 236783, 109, 20791, 236787, 3199, 611, 1900, 496, 19406, 573, 786, 236881, 107, 11020, 4169, 9071, 236787, 564, 236789, 236757, 15025, 236764, 840, 564, 740, 236789, 236745, 6361, 607, 600, 236761, 3551, 1873, 15858, 2174, 786, 531, 3927, 573, 6387, 1938, 2721, 580, 506, 3822, 236761, 

In [60]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class FunctionCallDataset(Dataset):
    def __init__(self, tokenized_list):
        self.data = tokenized_list
    
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return torch.tensor(self.data[idx], dtype=torch.long)

In [61]:
function_call_dataset = FunctionCallDataset(tokenized_dataset)

In [64]:
len(function_call_dataset.data)

96016

In [66]:
tokenizer.pad_token_id, tokenizer.eos_token_id, tokenizer.bos_token_id

(0, 1, 2)

In [69]:
tokenizer.decode(tokenizer.pad_token_id)

'<pad>'

In [70]:
tokenizer.vocab_size

262144

In [71]:
def dynamic_collate_fn(batch):
    pad_token_id = tokenizer.pad_token_id
    
    # 1. Pad the sequences so they all match the longest one in this batch
    padded_input_ids = pad_sequence(batch, batch_first=True, padding_value=pad_token_id)
    
    # 2. For Causal LM, labels are a direct copy of the input_ids
    labels = padded_input_ids.clone()
    
    # 3. Dynamically create the attention mask
    # It creates a tensor of 1s where the token is real, and 0s where it is padding
    attention_masks = (padded_input_ids != pad_token_id).long()

    # 4. Replace padding token ids in labels with -100 to ignore them in loss calculation
    labels[labels == pad_token_id] = -100
    
    # 3. Return the neat dictionary that standard NLP models expect
    return {
        'input_ids': padded_input_ids,
        'attention_mask': attention_masks,
        'labels': labels
    }

In [72]:
train_dataloader = DataLoader(
    function_call_dataset, 
    batch_size=8, 
    shuffle=True, 
    collate_fn=dynamic_collate_fn
)

In [73]:
first_batch = next(iter(train_dataloader))

In [74]:
first_batch['input_ids'].shape, first_batch['attention_mask'].shape, first_batch['labels'].shape

(torch.Size([8, 288]), torch.Size([8, 288]), torch.Size([8, 288]))

In [79]:
print(tokenizer.decode(first_batch['input_ids'].tolist()[4]))

<bos>SYSTEM: You are a helpful assistant with access to the following functions. Use them if required -
{
    "name": "search_definition",
    "description": "Search for the definition of a word",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The word or phrase to search for"
            }
        },
        "required": [
            "query"
        ]
    }
}


USER: What is the definition of the word "serendipity"?
ASSISTANT: <functioncall> {"name": "search_definition", "arguments": '{"query": "serendipity"}'} <|endoftext|><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><p

In [80]:
from torch.optim import AdamW
from transformers import get_scheduler
from transformers import AutoTokenizer, AutoModelForCausalLM

# Standard learning rate for fine-tuning
learning_rate = 5e-5
num_epochs = 1

# Load model from hugging face
model_id = 'google/gemma-3-270m'
pretrained_model = AutoModelForCausalLM.from_pretrained(model_id)

optimizer = AdamW(pretrained_model.parameters(), lr=learning_rate)

# Calculate total training steps
num_training_steps = num_epochs * len(function_call_dataset)

# Set up the learning rate scheduler
lr_scheduler = get_scheduler(
    name="linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)

Loading weights: 100%|██████████| 236/236 [00:00<00:00, 7238.59it/s]


In [ ]:
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_training_steps), desc="Training")
pretrained_model.train() # Put model in training mode

losses = []

for epoch in range(num_epochs):
    for batch in train_dataloader:
        if batch['input_ids'].shape[1] > 511:
            continue

        # Move all tensors in the batch to the designated device
        batch = {k: v for k, v in batch.items()}
        print(batch)
        
        # 1. Forward pass
        outputs = pretrained_model(**batch)
        loss = outputs.loss
        
        # 2. Backward pass
        loss.backward()
        losses.append(loss)
        
        # 3. Update weights and learning rate
        optimizer.step()
        lr_scheduler.step()
        
        # 4. Clear gradients for the next step
        optimizer.zero_grad()
        
        # Update progress bar and print loss
        progress_bar.update(1)
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})


Training:   0%|          | 0/96016 [00:00<?, ?it/s]